# 47 â€” Multi-Task MLP

PyTorch multi-task MLP with a shared backbone and many auxiliary heads:
pEC50 (primary), Emax, pEC50_null, physchem properties, and cliff membership flags.
NaN-masked loss allows partial-coverage tasks.

**Primary metric:** RAE (lower is better). Current best OOF RAE: 0.5281.

In [1]:
import os as _os
_torch_lib = r"d:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\torch\lib"
if _os.path.exists(_torch_lib):
    _os.add_dll_directory(_torch_lib)

import sys, os
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from pxr.data import load_train, load_test, load_counter
from pxr.featurize import combined, impute, morgan, rdkit_desc
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                   reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load data + all auxiliary targets

In [2]:
tr = load_train()
te = load_test()
print(f"Train: {len(tr)} | Test: {len(te)}")

tr['scaffold'] = tr['smiles'].apply(bemis_murcko)

# Combined features
X_tr_raw = combined(tr['smiles'].tolist())
X_te_raw = combined(te['smiles'].tolist())
X_tr_raw = impute(X_tr_raw)
X_te_raw = impute(X_te_raw)
print(f"X_tr: {X_tr_raw.shape}")

Train: 4139 | Test: 513


X_tr: (4139, 2265)


In [3]:
# Load counter-assay for pEC50_null
try:
    counter = load_counter()
    counter_lookup = counter.set_index('name')['pec50'].to_dict()
    tr['pec50_null'] = tr['name'].map(counter_lookup)
    print(f"Counter-assay matched: {tr['pec50_null'].notna().sum()} / {len(tr)}")
except Exception as e:
    print(f"Could not load counter-assay: {e}")
    tr['pec50_null'] = np.nan

# Physchem features (already in combined, but extract as separate regression targets)
print("Computing physchem targets...")
physchem_rows = tr['smiles'].apply(compute_physchem)
physchem_df = pd.DataFrame(physchem_rows.tolist(), index=tr.index)
for col in ['logp', 'tpsa', 'mw', 'hbd', 'hba']:
    if col in physchem_df.columns:
        tr[f'phys_{col}'] = physchem_df[col]
    else:
        tr[f'phys_{col}'] = np.nan
print("Physchem columns added.")

Counter-assay matched: 2647 / 4139
Computing physchem targets...


Physchem columns added.


In [4]:
# Cliff labels
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_path.exists():
    cliff_labels = pd.read_parquet(cliff_path)
    tr = tr.merge(cliff_labels[['name','is_cliff_member','cliff_role','max_cliff_delta']],
                  on='name', how='left')
    tr['is_cliff_member'] = tr['is_cliff_member'].fillna(False)
    tr['cliff_role'] = tr['cliff_role'].fillna(0)
else:
    fps = morgan_fp_batch(tr.smiles.tolist()).astype(np.float32)
    dot = fps @ fps.T
    rowsum = fps.sum(1)
    union = rowsum[:, None] + rowsum[None, :] - dot
    tanimoto_mat = dot / union.clip(min=1)
    np.fill_diagonal(tanimoto_mat, 0)
    tr['is_cliff_member'] = False
    tr['cliff_role'] = 0
    pec50_arr = tr['pec50'].values
    rows_cl, cols_cl = np.where((tanimoto_mat >= 0.6))
    for i, j in zip(rows_cl, cols_cl):
        if i >= j:
            continue
        if abs(pec50_arr[i] - pec50_arr[j]) >= 1.0:
            tr.loc[tr.index[i], 'is_cliff_member'] = True
            tr.loc[tr.index[j], 'is_cliff_member'] = True
            if pec50_arr[i] > pec50_arr[j]:
                tr.loc[tr.index[i], 'cliff_role'] = 1
                tr.loc[tr.index[j], 'cliff_role'] = -1
            else:
                tr.loc[tr.index[i], 'cliff_role'] = -1
                tr.loc[tr.index[j], 'cliff_role'] = 1

print(f"Cliff members: {tr['is_cliff_member'].sum()}")

Cliff members: 248


In [5]:
# Build multi-task target matrix
# Regression tasks (8): pec50, emax, pec50_null, logp, tpsa, mw, hbd, hba
# Classification tasks (2): is_cliff_member (binary), cliff_role>0 (binary)

reg_task_cols = ['pec50', 'emax', 'pec50_null',
                 'phys_logp', 'phys_tpsa', 'phys_mw', 'phys_hbd', 'phys_hba']
# Some columns may not exist
for col in reg_task_cols:
    if col not in tr.columns:
        tr[col] = np.nan

y_reg = tr[reg_task_cols].values.astype(np.float32)  # (N, 8)
y_cls = np.stack([
    tr['is_cliff_member'].astype(np.float32).values,
    (tr['cliff_role'] > 0).astype(np.float32).values,
], axis=1)  # (N, 2)

# Task weights: pEC50=4, emax=1, pec50_null=2, physchem=0.5 each, cliff tasks=2
REG_WEIGHTS  = np.array([4.0, 1.0, 2.0, 0.5, 0.5, 0.5, 0.5, 0.5], dtype=np.float32)
CLS_WEIGHTS  = np.array([2.0, 2.0], dtype=np.float32)

print("Multi-task target matrix:")
for i, col in enumerate(reg_task_cols):
    n_valid = np.sum(~np.isnan(y_reg[:, i]))
    print(f"  {col}: {n_valid}/{len(tr)} ({100*n_valid/len(tr):.0f}%) | weight={REG_WEIGHTS[i]}")
print(f"  is_cliff_member: {int(y_cls[:,0].sum())}/{len(tr)} | weight=2.0")
print(f"  cliff_active:    {int(y_cls[:,1].sum())}/{len(tr)} | weight=2.0")

Multi-task target matrix:
  pec50: 4139/4139 (100%) | weight=4.0
  emax: 4139/4139 (100%) | weight=1.0
  pec50_null: 2647/4139 (64%) | weight=2.0
  phys_logp: 4139/4139 (100%) | weight=0.5
  phys_tpsa: 4139/4139 (100%) | weight=0.5
  phys_mw: 4139/4139 (100%) | weight=0.5
  phys_hbd: 4139/4139 (100%) | weight=0.5
  phys_hba: 4139/4139 (100%) | weight=0.5
  is_cliff_member: 248/4139 | weight=2.0
  cliff_active:    133/4139 | weight=2.0


## 2. Architecture

In [6]:
class MultiTaskMLP(nn.Module):
    def __init__(self, d_in=2265, d_hidden=512, n_reg_tasks=8, n_cls_tasks=2):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.LayerNorm(d_hidden), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(d_hidden, d_hidden), nn.LayerNorm(d_hidden), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(d_hidden, 256), nn.LayerNorm(256), nn.GELU()
        )
        self.reg_heads = nn.ModuleList([nn.Linear(256, 1) for _ in range(n_reg_tasks)])
        self.cls_heads = nn.ModuleList([nn.Linear(256, 1) for _ in range(n_cls_tasks)])

    def forward(self, x):
        h = self.shared(x)
        reg = torch.stack([head(h).squeeze(-1) for head in self.reg_heads], dim=1)  # (B, n_reg)
        cls = torch.stack([torch.sigmoid(head(h).squeeze(-1)) for head in self.cls_heads], dim=1)  # (B, n_cls)
        return reg, cls


print("MultiTaskMLP defined.")
d_in = X_tr_raw.shape[1]
test_model = MultiTaskMLP(d_in=d_in)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Model parameters: {n_params:,}")

MultiTaskMLP defined.
Model parameters: 1,559,306


## 3. NaN-masked loss + training utilities

In [7]:
def masked_mse(pred, target):
    """MSE over non-NaN positions only."""
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0., device=pred.device)
    return ((pred[mask] - target[mask]) ** 2).mean()


def masked_bce(pred, target):
    """BCE over non-NaN positions."""
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0., device=pred.device)
    return nn.functional.binary_cross_entropy(pred[mask], target[mask])


def compute_multitask_loss(reg_pred, cls_pred, y_reg_batch, y_cls_batch,
                           reg_weights, cls_weights):
    reg_loss = sum(
        reg_weights[k] * masked_mse(reg_pred[:, k], y_reg_batch[:, k])
        for k in range(reg_pred.shape[1])
    )
    cls_loss = sum(
        cls_weights[k] * masked_bce(cls_pred[:, k], y_cls_batch[:, k])
        for k in range(cls_pred.shape[1])
    )
    return reg_loss + cls_loss


def train_multitask(
    X_train, y_reg_train, y_cls_train,
    reg_weights=REG_WEIGHTS, cls_weights=CLS_WEIGHTS,
    n_epochs=150, batch_size=128, lr=2e-3, device=DEVICE,
):
    d_in = X_train.shape[1]
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_train).astype(np.float32)

    model = MultiTaskMLP(d_in=d_in, d_hidden=512,
                         n_reg_tasks=y_reg_train.shape[1],
                         n_cls_tasks=y_cls_train.shape[1]).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    X_t = torch.tensor(X_sc, dtype=torch.float32)
    yr_t = torch.tensor(y_reg_train, dtype=torch.float32)
    yc_t = torch.tensor(y_cls_train, dtype=torch.float32)
    ds = TensorDataset(X_t, yr_t, yc_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_loss = float('inf')
    best_state = None

    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for x_b, yr_b, yc_b in loader:
            x_b, yr_b, yc_b = x_b.to(device), yr_b.to(device), yc_b.to(device)
            optimizer.zero_grad()
            reg_pred, cls_pred = model(x_b)
            loss = compute_multitask_loss(reg_pred, cls_pred, yr_b, yc_b,
                                          reg_weights, cls_weights)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 50 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}  loss={epoch_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, scaler


@torch.no_grad()
def predict_multitask(model, scaler, X, batch_size=256, device=DEVICE):
    model.eval()
    X_sc = scaler.transform(X).astype(np.float32)
    X_t = torch.tensor(X_sc, dtype=torch.float32)
    preds = []
    for i in range(0, len(X_t), batch_size):
        batch = X_t[i:i+batch_size].to(device)
        reg_pred, _ = model(batch)
        preds.append(reg_pred[:, 0].cpu().numpy())  # pEC50 is task 0
    return np.concatenate(preds)

print("Training utilities defined.")

Training utilities defined.


## 4. Scaffold 5-fold CV

In [8]:
y_tr_pec50 = tr['pec50'].values.astype(np.float32)
splits = scaffold_kfold_indices(tr['scaffold'], n_splits=5)
oof_preds = np.full(len(tr), np.nan)
fold_raes = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    print(f"\nFold {fold+1}/5 â€” train={len(tr_idx)}, val={len(val_idx)}")

    X_fold_tr = X_tr_raw[tr_idx]
    X_fold_val = X_tr_raw[val_idx]
    yr_fold_tr = y_reg[tr_idx]
    yc_fold_tr = y_cls[tr_idx]
    y_fold_val_pec50 = y_tr_pec50[val_idx]

    model, scaler = train_multitask(
        X_fold_tr, yr_fold_tr, yc_fold_tr,
        n_epochs=150, batch_size=128, lr=2e-3,
    )

    val_preds = predict_multitask(model, scaler, X_fold_val)
    fold_rae = rae(y_fold_val_pec50, val_preds)
    fold_raes.append(fold_rae)
    oof_preds[val_idx] = val_preds
    print(f"  Fold RAE: {fold_rae:.4f}")

oof_rae = rae(y_tr_pec50, oof_preds)
print(f"\n=== OOF RAE: {oof_rae:.4f} (mean fold: {np.mean(fold_raes):.4f} Â± {np.std(fold_raes):.4f}) ===")


Fold 1/5 â€” train=3311, val=828


    Epoch 50/150  loss=2385.1791


    Epoch 100/150  loss=1035.9538


    Epoch 150/150  loss=870.9381
  Fold RAE: 0.5188

Fold 2/5 â€” train=3311, val=828


    Epoch 50/150  loss=2467.2348


...

    Epoch 100/150  loss=1124.9660


    Epoch 150/150  loss=915.3646
  Fold RAE: 0.5908

Fold 5/5 â€” train=3312, val=827


    Epoch 50/150  loss=2431.3554


    Epoch 100/150  loss=1169.1546


    Epoch 150/150  loss=1028.7586
  Fold RAE: 0.6334

=== OOF RAE: 0.5882 (mean fold: 0.5933 Â± 0.0398) ===


## 5. Final model â€” train on all data, predict test

In [9]:
print("Training final model on all training data...")
final_model, final_scaler = train_multitask(
    X_tr_raw, y_reg, y_cls,
    n_epochs=150, batch_size=128, lr=2e-3,
)

test_preds = predict_multitask(final_model, final_scaler, X_te_raw)

y_lo = y_tr_pec50.min() - 0.5
y_hi = y_tr_pec50.max() + 0.5
test_preds = np.clip(test_preds, y_lo, y_hi)
print(f"Test pred range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")

Training final model on all training data...


    Epoch 50/150  loss=2018.1365


    Epoch 100/150  loss=1072.0149


    Epoch 150/150  loss=888.9547
Test pred range: [2.179, 5.735]


In [10]:
# Save OOF
np.save(DATA_PROCESSED / 'oof_multitask_mlp.npy', oof_preds)
print("Saved OOF predictions.")

# Save submission
sub = pd.DataFrame({'Molecule Name': te['name'], 'pEC50': test_preds})
out_path = SUBMISSIONS / '47_multitask_mlp.csv'
sub.to_csv(out_path, index=False)
print(f"Saved submission to {out_path}")
print(sub.head())
print(f"\nFinal OOF RAE: {oof_rae:.4f}")

Saved OOF predictions.
Saved submission to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\47_multitask_mlp.csv
    Molecule Name     pEC50
0  OADMET-0006617  4.665734
1  OADMET-0006616  4.528043
2  OADMET-0006615  4.296251
3  OADMET-0006614  5.195253
4  OADMET-0006613  4.866846

Final OOF RAE: 0.5882
